In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
import shapely # map-rendering library
import numpy as np
import fiona # enging for reading more robust data files
import contextily as cx

In [ ]:
# Reading in counties data
counties = gpd.read_file("../Parks/Counties.geojson", engine="fiona")
# counties.head()
# Reading in Parks data
parks = gpd.read_file("../Parks/Parks_(2024).geojson")
# parks.head()
# Changing Data to Coordinate System for Texas
counties_projected = counties.to_crs("EPSG:3857")
parks_projected = parks.to_crs("EPSG:3857")
# counties_projected.head()

In [ ]:
# Gets the geometry for Dallas County alone
dallas_geometry = counties_projected.loc[counties_projected["COUNTY"] == "Dallas", "geometry"].iloc[0]
# Find for each park, if it intersects with Dallas County
parks_projected["intersects"] = parks_projected.intersects(dallas_geometry) 

In [ ]:
# Shows which parks intersect with Dallas County
parks_projected.plot(column="intersects", categorical=True, legend=True);

In [ ]:
# Subsets data into new data frame, only parks that intersect with dallas county
dallas_county_parks = parks_projected.loc[parks_projected["intersects"] == True]
dallas_county_parks.plot(color = "darkolivegreen")

In [ ]:
# The dataset contains proposed and existing parks. We only want the ones that actually exist
# Check to see the different categories
print(dallas_county_parks["STATUS"].value_counts())
# Subset the data only for existing parks
dallas_county_parks = dallas_county_parks[dallas_county_parks["STATUS"] == "Existing"]
# Check that this was done correctly
dallas_county_parks["STATUS"].value_counts()

In [ ]:
# For mapping only
dallas_county_parks_map = dallas_county_parks
# Calculate centroids
dallas_county_parks_map["centroid"] = dallas_county_parks_map.centroid

ax = dallas_county_parks_map.plot(color="darkolivegreen", edgecolor="black", figsize=(10, 10))
ax = dallas_county_parks_map.set_geometry("centroid").plot(ax=ax, color="red", markersize=5)
cx.add_basemap(ax)
ax.set_title("Dallas County Parks with Centroids")
ax.set_axis_off();

In [ ]:
# Want to test a specific case for the Trinity River Greenbelt (the big green area in the middle) - does the centroid fall within the area of the park itself?
dallas_county_parks_map[dallas_county_parks_map["NAME"] == "Trinity River Greenbelt"]

In [ ]:
# To test, make a color column with everything red
dallas_county_parks_map["color"] = "red"
# Then make the Greenbelt rows a different color so that we can see it
dallas_county_parks_map.loc[1918, "color"] = "blue"
dallas_county_parks_map.loc[2604, "color"] = "pink"

In [ ]:
# We can see that the blue dot is outside of the actual park area, while the pink one is
ax = dallas_county_parks_map.plot(color="darkolivegreen", edgecolor="black", figsize=(10, 10))
ax = dallas_county_parks_map.set_geometry("centroid").plot(ax=ax, color=dallas_county_parks_map["color"], markersize=5)
cx.add_basemap(ax)
ax.set_title("Dallas County Parks with Centroids")
ax.set_axis_off();

In [ ]:
# Project for calculating distances
dallas_county_parks = dallas_county_parks.to_crs("EPSG:4326")
dallas_county_parks["centroids"] = dallas_county_parks.centroid

# Looks Correct
# ax = dallas_county_parks.plot(color="darkolivegreen", edgecolor="black", figsize=(10, 10))
# ax = dallas_county_parks.set_geometry("centroids").plot(ax=ax, color="red", markersize=5)
# cx.add_basemap(ax)
# ax.set_title("Dallas County Parks with Centroids")
# ax.set_axis_off();

# Take only the columns that we need
centroids = dallas_county_parks[["NAME", "CITY", "centroids"]]
# Export the data as a shapefile
centroids.to_file('centroids.shp')